In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str

In [ ]:
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
def generate_joke(state:JokeState):
    prompt=f"Generate a joke on the topic : {state['topic']}"
    
    response=llm.invoke(prompt).content[0]['text']
    
    return {"joke":response}

In [ ]:
def explain_joke(state:JokeState):
    prompt=f"Explain the following joke in detail : {state['joke']}"
    
    response=llm.invoke(prompt).content[0]['text']
    return {"explaination":response}

In [ ]:
graph=StateGraph(JokeState)

graph.add_node("generate_joke",generate_joke)
graph.add_node("explain_joke",explain_joke)

graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke","explain_joke")
graph.add_edge("explain_joke",END)

checkpoint=InMemorySaver()
workflow=graph.compile(checkpointer=checkpoint)

In [ ]:
configuration={"configurable":{"thread_id":1}}
final_state=workflow.invoke({"topic":"AI"},config=configuration)

In [ ]:
final_state